# AI Golf Caddy — Fork, set up, and use

Short checklist: **fork → run locally** → **host with Supabase + Render + Vercel** → **ElevenLabs for voice** → **use the app**. Full copy also lives in **`README.md`**.

---

## Prerequisites (accounts + tools)

| Need | Why |
|------|-----|
| **Git** | Fork the repo, clone your fork. |
| **Python 3.11+** | FastAPI backend (`backend/`). |
| **Node.js 20+**, **npm** | Next.js frontend (`frontend/`). |
| [**Supabase**](https://supabase.com) | Hosted **Postgres** → `DATABASE_URL`. |
| [**Anthropic**](https://console.anthropic.com) | **Claude** for “Talk with caddie”. |
| [**ElevenLabs**](https://elevenlabs.io) | **Text-to-speech** for “Listen (ElevenLabs)” → `ELEVENLABS_API_KEY` + a **Voice ID**. |
| [**Render**](https://render.com) (prod) | Host the **Docker** API from `backend/Dockerfile`. |
| [**Vercel**](https://vercel.com) (prod) | Host the **Next.js** app from `frontend/`. |

---

## 1. Fork and clone

1. On GitHub: **Fork** the repository.  
2. Clone **your fork**:

```bash
git clone https://github.com/<YOUR_USERNAME>/golf_caddy_app.git
cd golf_caddy_app
```

Work in **`backend/`** (API) and **`frontend/`** (UI).

---

## 2. Supabase (database)

1. Dashboard → **New project** → note the **database password**.  
2. **Project Settings → Database** → copy the **URI** (connection string), e.g. `postgresql://postgres...`  
3. Use that as **`DATABASE_URL`** in your backend environment (local shell or **Render → Environment**).

On first API start, migrations run and create the tables the app uses.

---

## 3. Backend locally (FastAPI + optional ElevenLabs)

```bash
cd backend
python3 -m venv .venv
source .venv/bin/activate          # Windows: .venv\Scripts\activate
pip install -r requirements.txt

# Required
export DATABASE_URL="postgresql://..."          # Supabase
export ANTHROPIC_API_KEY="sk-ant-api03-..."     # Claude (advice)
export CORS_ALLOW_ORIGINS="http://localhost:3000"

# Optional — enables “Listen (ElevenLabs)” in the caddie modal
export ELEVENLABS_API_KEY="..."                 # elevenlabs.io → Profile / API keys
export ELEVENLABS_VOICE_ID="..."                # Voices → copy Voice ID
# export ELEVENLABS_MODEL_ID="eleven_multilingual_v2"   # optional override (default in code)

uvicorn app.main:app --reload --port 8000
```

API base: **`http://127.0.0.1:8000`** — try **`GET /api/health`**.

---

## 4. Frontend locally (Next.js → Vercel later)

```bash
cd frontend
npm install
cp .env.example .env.local
```

In **`.env.local`** point the browser at your API:

```
NEXT_PUBLIC_API_BASE_URL=http://localhost:8000
```

> **Alternate:** leave `NEXT_PUBLIC_API_BASE_URL` empty and set **`BACKEND_API_BASE_URL=http://localhost:8000`** so Next **rewrites** `/api/*` to the backend (`next.config.mjs`).

```bash
npm run dev
```

Open **`http://localhost:3000`**.

---

## 5. Quick check (optional)

Run the next cell while **`uvicorn`** is up, or open the URL in a browser.

In [ ]:
import json
from urllib import error, request

url = "http://127.0.0.1:8000/api/health"
try:
    with request.urlopen(url, timeout=2) as r:
        print(json.loads(r.read().decode()))
except error.URLError as e:
    print("Backend not running or wrong URL:", e)

## 6. Production: Supabase + Render + Vercel + ElevenLabs

### Supabase (already above)
- Same **`DATABASE_URL`** on **Render** as in local dev.

### Render (backend API)
1. [**Render Dashboard**](https://dashboard.render.com) → **New +** → **Web Service**.  
2. Connect your GitHub repo (your fork).  
3. Configure:
   - **Root directory:** `backend` (or Docker build context that contains `backend/Dockerfile`).  
   - **Dockerfile path:** `Dockerfile` (if root is `backend`, this is `backend/Dockerfile` in the repo).  
4. Under **Environment**, add **secrets**:

| Variable | Purpose |
|----------|--------|
| **`DATABASE_URL`** | Supabase Postgres URI |
| **`ANTHROPIC_API_KEY`** | Claude for `/api/caddie/advice` |
| **`CORS_ALLOW_ORIGINS`** | Your live site, e.g. `https://your-app.vercel.app` (comma-separated if several) |
| **`ELEVENLABS_API_KEY`** | ElevenLabs TTS for `/api/caddie/tts` |
| **`ELEVENLABS_VOICE_ID`** | Default voice for TTS (Voices page in ElevenLabs) |

5. Deploy → copy the public URL (e.g. `https://your-api.onrender.com`).  
6. Verify: `https://your-api.onrender.com/api/health`

### ElevenLabs
- Sign up at [**elevenlabs.io**](https://elevenlabs.io) → create an **API key**.  
- **Voices** → pick a voice → copy **Voice ID** into **`ELEVENLABS_VOICE_ID`** on **Render** (and locally if testing TTS).  
- Without these, **advice still works**; only **Listen (ElevenLabs)** returns an error until keys are set.

### Vercel (frontend)
1. [**Vercel**](https://vercel.com) → **Add New… → Project** → import the same repo.  
2. **Root Directory:** `frontend`  
3. **Environment variable:**
   - **`NEXT_PUBLIC_API_BASE_URL`** = `https://your-api.onrender.com` (no trailing slash)
4. Deploy → open `https://your-project.vercel.app`  
5. If the browser shows **CORS** errors on `/api/...`, ensure **`CORS_ALLOW_ORIGINS`** on Render includes **`https://your-project.vercel.app`** exactly (scheme + host).

---

## 7. How to use the app

### Start
- Tap **Play**. Choose **Live** (GPS) or **Sim** (drag position on the map).  
- Grant **location** for Live so distances and advice match where you stand.

### Course / hole
- Select **course** and **hole** (arrows or hole picker).  
- Watch **distance (adjusted)** and wind in the header when data has loaded.

### Map
- **Blue dot** = you. **White marker** = aim / bend (drag to adjust).  
- After **Talk with caddie**, the app may call **`/api/caddie/suggest-target`** once; you can still drag the marker.

### Talk with caddie
- Opens a modal with **summary** text and collapsible **briefing** lines.  
- **Listen (ElevenLabs)** — needs **`ELEVENLABS_API_KEY`** and **`ELEVENLABS_VOICE_ID`** on the **backend** (Render or local).  
- **Device voice** — uses the browser if you prefer not to call ElevenLabs.  
- **Refresh advice** — fetch new text from Claude.

### Settings & scorecard
- **`/settings`** — bag carry distances and shot-shape buckets (helps club recommendations).  
- **Scorecard** — track strokes per hole.


---
*End.*